In [1]:
!pip install -U ipywidgets roboflow ultralytics

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("roboflow_api_key")
secret_value_0 = user_secrets.get_secret("wandb_api_key")

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=secret_value_1)
project = rf.workspace("4564-sdivc").project("helmet-etb3q")
version = project.version(1)
dataset = version.download("yolov11")

In [ ]:
project = rf.workspace("miniproject-dfm83").project("helmetdetectiondataset")
version = project.version(7)
dataset = version.download("yolov8")

In [ ]:
# Visualize dataset
import cv2
import matplotlib.pyplot as plt
import os

# Path to your images and annotations
image_dir = '/kaggle/working/helmet-1/train/images'
label_dir = '/kaggle/working/helmet-1/train/labels'

# Get a list of image files
image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

# Visualize a few images with bounding boxes
for image_file in image_files[:5]:  # Display only the first 5 images for brevity
    # Read image
    image_path = os.path.join(image_dir, image_file)
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    error = []
    
    # Read corresponding annotation file
    annotation_path = os.path.join(label_dir, image_file.replace('.jpg', '.txt'))
    if os.path.exists(annotation_path):
        with open(annotation_path, 'r') as file:
            for line in file.readlines():
                parts = line.strip().split()
                if len(parts) <= 5:
                    class_id, x_center, y_center, width, height = map(float, line.strip().split())
                    
                    # Convert normalized coordinates to pixel coordinates
                    img_height, img_width = img.shape[:2]
                    x_center *= img_width
                    y_center *= img_height
                    width *= img_width
                    height *= img_height
                    
                    # Calculate top-left corner
                    top_left_x = int(x_center - width / 2)
                    top_left_y = int(y_center - height / 2)
                    bottom_right_x = int(x_center + width / 2)
                    bottom_right_y = int(y_center + height / 2)
                    
                    # Draw bounding box
                    cv2.rectangle(img, (top_left_x, top_left_y), (bottom_right_x, bottom_right_y), (255, 0, 0), 2)
                    cv2.putText(img, f'Class {int(class_id)}', (top_left_x, top_left_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
                else:
                    print(image_path)
                    print(annotation_path)
    # Display the image
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')
    
    plt.show()

In [ ]:
from ultralytics import YOLO

# Load a YOLO model
model = YOLO("yolov10m.pt")

# Train and Fine-Tune the Model
model.train(data="/kaggle/working/HelmetDetectionDataset-7/data.yaml", epochs=200, project="helmet-detector", name="yolo10m")


100%|██████████| 32.1M/32.1M [00:00<00:00, 61.4MB/s]


Ultralytics 8.3.31 🚀 Python-3.10.14 torch-2.4.0 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=yolov10m.pt, data=/kaggle/working/HelmetDetectionDataset-7/data.yaml, epochs=200, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=helmet-detector, name=yolo10m, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False,

train: Scanning /kaggle/working/HelmetDetectionDataset-7/train/labels... 785 images, 0 backgrounds, 0 corrupt: 100%|██████████| 785/785 [00:00<00:00, 1127.48it/s]


train: New cache created: /kaggle/working/HelmetDetectionDataset-7/train/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 333, len(boxes) = 2906. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))


/opt/conda/lib/python3.10/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 1.4.21 (you have 1.4.17). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
val: Scanning /kaggle/working/HelmetDetectionDataset-7/valid/labels... 212 images, 0 backgrounds, 0 corrupt: 100%|██████████| 212/212 [00:00<00:00, 943.65it/s]

...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    134/200      10.3G      1.636      1.106      2.137          1        640: 100%|██████████| 50/50 [00:34<00:00,  1.44it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:03<00:00,  2.20it/s]

                   all        212        643      0.706      0.618      0.675      0.472



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    135/200      10.3G      1.608      1.059      2.174         93        640:  10%|█         | 5/50 [00:03<00:31,  1.41it/s]

In [ ]:
%tensorboard --logdir helmet-detector/yolo10m

In [ ]:
from ultralytics import YOLO  # Uncomment this line if needed
import cv2
import os

# Load the model
model = YOLO("/kaggle/working/helme-detector/yolo11n/weights/best.pt")
output_dir = 'kaggle/working/runs'
# Define class IDs
MOTORBIKE_CLASS_ID = 1
NO_HELMET_CLASS_ID = 3

# Predict on the video
results = model.predict(source='/kaggle/input/motorbike-helmet-dataset/input/videos/input2.mp4', stream=True, save=True, save_dir=output_dir)

In [ ]:
# Create a directory to save cropped images
output_dir = '/kaggle/working/inference_results/no_helmet_crops'
os.makedirs(output_dir, exist_ok=True)

frame_count = 0  # To keep track of frame numbers for unique image filenames

# Process each result
for r in results:
    frame = r.orig_img

    for box in r.boxes:
        class_id = int(box.cls[0])  # Get the class ID of the detected object

        if class_id == NO_HELMET_CLASS_ID:
            # Get the bounding box coordinates
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Crop the image
            cropped_img = frame[y1:y2, x1:x2]

            # Save the cropped image with a unique filename
            crop_filename = os.path.join(output_dir, f'no_helmet_{frame_count:04d}.jpg')
            cv2.imwrite(crop_filename, cropped_img)

            frame_count += 1

print(f"Cropped images of motorbikes without helmets saved in {output_dir}")

In [ ]:
import cv2
def convert_avi_to_mp4(input_path, output_path):
    """Convert an AVI video file to MP4 format using OpenCV."""
    cap = cv2.VideoCapture(input_path)
    
    # Get the video's properties (frame width, height, and FPS)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the codec and create a VideoWriter object for MP4 output
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for MP4 format
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Read and write each frame to the new video file
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)
        
    # Release resources
    cap.release()
    out.release()
    print(f"Video saved as {output_path}")

In [ ]:
import os

os.remove('/kaggle/working/helmet-detector.zip')

In [2]:
import shutil

# Specify the path of the folder you want to delete
folder_path = '/kaggle/working/helme-detector'

# Delete the folder and all its contents
shutil.rmtree(folder_path)

In [ ]:
import shutil

# Compress only specific subdirectory
shutil.make_archive('/kaggle/working/helmet-detector', 'zip', '/kaggle/working/helme-detector')